In [2]:
# docker-compose up -d


In [1]:
from pymilvus import connections
connections.connect(alias="default", host="127.0.0.1", port="19530")


In [2]:
from pymilvus import utility
print("Collections:", utility.list_collections())


Collections: ['pdf_docs_test', 'milvus_langchain_test']


In [3]:
# direct_gemini_to_milvus.py
import os
import uuid
import google.generativeai as genai            # google-generativeai package
from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection, utility


C:\Users\USER\Miniconda3\envs\kpn-ai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# 0. Configure API key
genai.configure(api_key=os.environ["GEMINI_API_KEY"])

In [5]:
import pdfplumber

In [13]:
def extract_content_from_pdf(pdf_path):
    with pdfplumber.open(pdf_path) as pdf:
        all_content = []
        for page_num, page in enumerate(pdf.pages, start=1):
            page_data = {"text": None, "tables": [], 
                         "meta":{"page": page_num, "source":os.path.basename(pdf_path)}}
            
            # 1. Extract free text
            text = page.extract_text()
            if text:
                page_data["text"] = text.strip()
            
            # 2. Extract ALL tables (not just one)
            tables = page.extract_tables()
            if tables:
                page_data["tables"] = tables
            
            all_content.append(page_data)
    return all_content


In [11]:
folder_path = 'assets'

content = []
for filename in os.listdir(folder_path):
    # Construct the full filepath
    file_path = os.path.join(folder_path, filename)
    print(file_path)
    content.append(extract_content_from_pdf(file_path))
    


assets\CP-Bantuan-Pengobatan-2025.pdf


In [12]:
content

[{'id': 1,
  'text': 'KPN CORP\nBANTUAN PENGOBATAN\nFEB 2025\nDEFINITION\n● BANTUAN PENGOBATAN adalah bantuan biaya kesehatan di luar BPJS Kesehatan\nyang diberikan Perusahaan kepada karyawan beserta keluarganya yang sakit dan\nmemerlukan pengobatan oleh tenaga medis atau dokter yang meliputi Bantuan\nPengobatan Rawat Jalan, Bantuan Pengobatan Rawat Inap, Bantuan Biaya Persalinan,\nBantuan Kacamata, dan Bantuan Pengobatan Kesehatan Khusus. 1\n● BANTUAN PENGOBATAN RAWAT JALAN adalah bantuan biaya kesehatan yang\ndiberikan Perusahaan kepada karyawan dan keluarganya yang sakit dan memerlukan\npengobatan oleh tenaga medis atau dokter tetapi tidak memerlukan rawat inap. 2\n● BANTUAN RAWAT INAP adalah bantuan biaya kesehatan yang diberikan Perusahaan\nkepada karyawan dan keluarganya yang sakit serta berdasarkan rekomendasi dan\ndiagnosa dokter memerlukan perawatan inap di rumah sakit atau fasilitas medis lainnya\nyang ditentukan oleh dokter yang menangani masalah kesehatan tersebut. 3\n● BAN

In [21]:
# 2. Create embeddings in batch (respect rate limits)
def get_gemini_embeddings(texts, model="models/embedding-001"):
    # genai.embed_content returns embeddings -- SDK shape can vary by version.
    # This example assumes embed_content returns a list/dict with 'embedding' vectors.
    response = genai.embed_content(model=model, content=texts)
#     print(response)
    # response structure may be a list; adapt if your SDK version differs
    embeddings = []
    # If response is a dict with 'data' or 'predictions', inspect structure and extract vectors.
    # Example based on typical SDK:
    for item in response["embedding"]:
        embeddings.append(item)
    return embeddings

In [10]:
# 3. Connect to Milvus
connections.connect(host="127.0.0.1", port="19530")

In [11]:
# 4. Create Collection (only once)
collection_name = "pdf_docs_test"
if utility.has_collection(collection_name):
    collection = Collection(collection_name)
else:
    dim = 1536  # set to your embedding dimension (check model); adjust accordingly
    fields = [
        FieldSchema(name="id", dtype=DataType.VARCHAR, is_primary=True, max_length=64),
        FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=dim),
        FieldSchema(name="meta", dtype=DataType.JSON)
    ]
    schema = CollectionSchema(fields, description="PDF doc embeddings")
    collection = Collection(name=collection_name, schema=schema)
    # create index (optional, example IVF_FLAT)
    from pymilvus import Index
    index_params = {"index_type": "IVF_FLAT", "metric_type": "COSINE", "params": {"nlist": 1024}}
    collection.create_index(field_name="embedding", index_params=index_params)


In [22]:

# 5. Prepare batches, create embeddings, insert
batch_size = 32
for i in range(0, len(documents), batch_size):
    batch = documents[i:i+batch_size]
    texts = [d["text"] for d in batch]
    embeddings = get_gemini_embeddings(texts)  # list of vectors
    ids = [d["id"] for d in batch]
    metas = [d["meta"] for d in batch]

    # Insert expects column-aligned lists (matching schema order)
    entities = [ids, embeddings, metas]
    collection.insert(entities)

# 6. Flush & load for search
collection.flush()
collection.load()
print("Inserted and loaded collection.")


2025-09-18 15:39:03,554 [ERROR][handler]: RPC error: [batch_insert], <MilvusException: (code=1100, message=the dim (768) of field data(embedding) is not equal to schema dim (1536): invalid parameter[expected=1536][actual=768])>, <Time:{'RPC start': '2025-09-18 15:39:03.545738', 'RPC error': '2025-09-18 15:39:03.554741'}> (decorators.py:140)


MilvusException: <MilvusException: (code=1100, message=the dim (768) of field data(embedding) is not equal to schema dim (1536): invalid parameter[expected=1536][actual=768])>